In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # chuẩn hoá Standard

In [2]:
df = pd.read_csv("../../.data/iris/iris_3class_big.csv")
print(df.shape)

print(df.head())

(1500, 5)
   sepal_length  sepal_width  petal_length  petal_width          species
0           7.8          3.9           5.8          1.6   Iris-virginica
1           5.2          2.6           3.8          1.4  Iris-versicolor
2           5.9          3.2           3.7          1.3  Iris-versicolor
3           5.1          3.8           1.9          0.6      Iris-setosa
4           4.7          4.3           1.5          0.3      Iris-setosa


In [3]:
x = df.iloc[:, :-1].values
y = df.iloc[:, -1]

print(y.head())
print(y.shape)

list_class = np.unique(y)
print(list_class)



0     Iris-virginica
1    Iris-versicolor
2    Iris-versicolor
3        Iris-setosa
4        Iris-setosa
Name: species, dtype: object
(1500,)
['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


In [4]:
encode = {
    name: num for num, name in enumerate(list_class)
}

print(encode)

{'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}


In [5]:
y = y.map(encode)

In [6]:
print(y)
print(x.shape)
print(y.shape)

0       2
1       1
2       1
3       0
4       0
       ..
1495    2
1496    1
1497    2
1498    1
1499    1
Name: species, Length: 1500, dtype: int64
(1500, 4)
(1500,)


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)

print(x_train.shape , y_train.shape)
print(x_test.shape, y_test.shape)

(1200, 4) (1200,)
(300, 4) (300,)


In [8]:
class MySoftmaxRegression:
    def __init__(self, lr = 0.01, esp = 1e-9, epochs = 1000, numClass = 2):
        self.lr = lr
        self.epochs = epochs
        self.numClass = numClass
        self.W = None
        self.b = None
        self.loss = []

    def softmax(self, z):
        z = z - np.max(z, axis = 1, keepdims = True)
        e = np.exp(z)
        return e / np.sum(e, axis=1, keepdims = True)

    def forward(self, X):
        return self.softmax(X@self.W + self.b)

    def loss_func(self, y, y_hat):
        return np.mean(-np.sum( y*np.log(y_hat) , axis = 1))

    def accuracy(self, y, y_hat):
        y_pred = np.argmax(y_hat, axis=1)
        y_true = np.argmax(y, axis=1)
        return np.mean(y_pred == y_true)

    def train(self, X, y):
        self.W = np.random.rand(X.shape[1], self.numClass)
        self.b = 0

        # chuẩn hoá y thành dãy chấm
        y = np.eye(self.numClass)[y]

        for epoch in range(self.epochs):
            y_hat = self.forward(X)
            loss_epoch = self.loss_func(y, y_hat)
            self.loss.append(loss_epoch)

            error = y_hat - y
            dw = (1 / X.shape[0]) * np.dot(X.T, error)
            db = (1 / X.shape[0]) * np.sum(error, axis=0, keepdims=True)

            self.W -= self.lr * dw
            self.b -= self.lr * db

            temp_acc = self.accuracy(y_hat, y)

            if epoch % 10 == 0:
                print(f"Epoch: {epoch}   | Acc: {temp_acc}  | Loss: {self.loss[-1]}")



In [9]:
myModel = MySoftmaxRegression(epochs = 100000, numClass=len(encode))
myModel.train(x_train, y_train)

y_hat = myModel.forward(x_test)
y_test = np.eye(len(encode))[y_test]

print(f"Tập test Loss = {myModel.loss_func(y_test, y_hat)} ")
print(f"Tập test Accuracy: {myModel.accuracy(y_test, y_hat)}")


Epoch: 0   | Acc: 0.3358333333333333  | Loss: 2.190620202422014
Epoch: 10   | Acc: 0.5016666666666667  | Loss: 1.031469062506234
Epoch: 20   | Acc: 0.43416666666666665  | Loss: 0.9632198702644605
Epoch: 30   | Acc: 0.43333333333333335  | Loss: 0.9134388917955069
Epoch: 40   | Acc: 0.43666666666666665  | Loss: 0.8699309564546986
Epoch: 50   | Acc: 0.5666666666666667  | Loss: 0.8318231078616503
Epoch: 60   | Acc: 0.7475  | Loss: 0.7984084993568534
Epoch: 70   | Acc: 0.7758333333333334  | Loss: 0.7690401916335372
Epoch: 80   | Acc: 0.7858333333333334  | Loss: 0.7431426317715804
Epoch: 90   | Acc: 0.7891666666666667  | Loss: 0.7202150031996221
Epoch: 100   | Acc: 0.7908333333333334  | Loss: 0.6998279729140227
Epoch: 110   | Acc: 0.7958333333333333  | Loss: 0.6816169623219803
Epoch: 120   | Acc: 0.8016666666666666  | Loss: 0.6652740822653566
Epoch: 130   | Acc: 0.805  | Loss: 0.65053999996179
Epoch: 140   | Acc: 0.8108333333333333  | Loss: 0.637196399502269
Epoch: 150   | Acc: 0.81333333333